# Método del codo para elegir n_neighbors (KNNImputer)

Tomamos las filas de **train** que no tienen ningún NaN (las "completas"), les borramos artificialmente una fracción de valores conocidos, los imputamos con distintos `k`, y medimos qué tan lejos quedó la reconstrucción del valor real (NRMSE). Se repite varias veces con distintas máscaras aleatorias para que la curva no dependa de una sola tirada de dados, y se promedia.

Todo esto se hace **solo con datos de entrenamiento** (nunca con test): elegir un hiperparámetro como `k` mirando el conjunto de test sería la misma familia de fuga de datos que vimos con el escalador y el imputador.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
import matplotlib.pyplot as plt

## Reproducir el pipeline hasta el escalado

Split, eliminación de las filas de outliers reales (solo si cayeron en train) y `RobustScaler` ajustado únicamente con train. Si ya tenés `X_train_esc` y `columnas_continuas` armados en tu notebook, podés saltear esta celda y usar los tuyos.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

df = pd.read_csv('house-prices-tp.csv')
df_limpio = df.dropna(subset=['MEDV']).copy()

filas_a_eliminar = [14, 23, 264, 298, 316, 458, 520, 554]
X = df_limpio.drop(columns='MEDV')
y = df_limpio['MEDV']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
filas_en_train = [i for i in filas_a_eliminar if i in X_train.index]
X_train = X_train.drop(index=filas_en_train)
y_train = y_train.drop(index=filas_en_train)

columnas_continuas = [c for c in X_train.columns if c not in ['CHAS', 'RAD']]
scaler = RobustScaler()
X_train_esc = X_train.copy()
X_train_esc[columnas_continuas] = scaler.fit_transform(X_train[columnas_continuas])

## Función: enmascarar y medir

Enmascara aleatoriamente una fracción de las celdas, imputa con `KNNImputer(n_neighbors=k)` y devuelve el NRMSE de la reconstrucción, calculado solo sobre las celdas enmascaradas (normalizado por el desvío de cada columna para poder promediar entre columnas de escalas distintas).

In [ ]:
def enmascarar_y_medir(datos_verdad, fraccion_mascara, k, rng):
    mascara = rng.random(datos_verdad.shape) < fraccion_mascara
    datos_enmascarados = datos_verdad.copy()
    datos_enmascarados[mascara] = np.nan

    imputer = KNNImputer(n_neighbors=k)
    reconstruido = imputer.fit_transform(datos_enmascarados)

    std_cols = datos_verdad.std(axis=0)
    error = (reconstruido[mascara] - datos_verdad[mascara]) ** 2
    std_expandido = np.tile(std_cols, (datos_verdad.shape[0], 1))[mascara]
    return np.sqrt(np.mean(error / (std_expandido ** 2 + 1e-12)))

## Función: curva del codo

Repite el experimento de enmascaramiento `n_repeticiones` veces para cada valor de `k`, y devuelve una tabla con la media y el desvío del NRMSE por `k`. Más repeticiones = curva más estable, menos ruido.

In [ ]:
def curva_del_codo(datos_verdad, ks, n_repeticiones=15, fraccion_mascara=0.05):
    filas = []
    for k in ks:
        errores = []
        for rep in range(n_repeticiones):
            rng = np.random.default_rng(rep)
            errores.append(enmascarar_y_medir(datos_verdad, fraccion_mascara, k, rng))
        filas.append({
            'k': k,
            'nrmse_medio': np.mean(errores),
            'nrmse_std': np.std(errores),
        })
    return pd.DataFrame(filas)

## Función: graficar el codo

NRMSE promedio (+/- desvío) vs `k`, para ubicar el codo a ojo.

In [ ]:
def graficar_codo(tabla_resultados):
    plt.figure(figsize=(9, 5))
    plt.errorbar(
        tabla_resultados['k'], tabla_resultados['nrmse_medio'],
        yerr=tabla_resultados['nrmse_std'], marker='o', capsize=3
    )
    plt.title('Método del codo para elegir n_neighbors (KNNImputer)')
    plt.xlabel('n_neighbors (k)')
    plt.ylabel('NRMSE promedio (experimento de enmascaramiento)')
    plt.grid(alpha=0.3)
    plt.show()

## Ejecutar el experimento

Se usa **solo** las filas completas (sin NaN) de `X_train_esc` — nunca se toca test para elegir el hiperparámetro.

In [ ]:
completas = X_train_esc[columnas_continuas].dropna().copy()
print(f"Filas completas en train usadas para el experimento: {completas.shape[0]}")

datos_verdad = completas.to_numpy()
ks_a_probar = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30]

tabla = curva_del_codo(datos_verdad, ks_a_probar, n_repeticiones=15, fraccion_mascara=0.05)
tabla

In [ ]:
graficar_codo(tabla)

## Interpretar el resultado

El `k` con menor NRMSE promedio es el **mínimo exacto** de la curva, pero no necesariamente es "el codo". El codo es el punto donde la curva deja de bajar fuerte — normalmente un poco antes del mínimo exacto. Si varios valores de `k` quedan dentro del mismo desvío estándar entre sí, conviene priorizar el más chico de esa zona plana (más conservador con la varianza original de los datos, siguiendo a Beretta & Santaniello).

In [ ]:
k_minimo = int(tabla.loc[tabla['nrmse_medio'].idxmin(), 'k'])
print(f"k con menor NRMSE promedio (mínimo exacto de la curva): {k_minimo}")